# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliRazaNiazi804/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Resources

This section sets up the necessary libraries and connects to the DuckDB database.

In [1]:
import duckdb
import pandas as pd
from google.colab import userdata

# Initialize DuckDB connection to a persistent database file
con = duckdb.connect(database=':memory:', read_only=False)

# Example of how to define TABLES if they are part of a specific schema/path
# In a real scenario, this would point to your actual data source.
TABLES = {
    'fact_daily': 'fact_daily' # Assuming 'fact_daily' is a table directly available in the connected DB
}

# For demonstration, let's create a dummy fact_daily table if it doesn't exist
# In your actual W03, this data would be loaded from your full release DuckDB setup.
con.execute("""
    CREATE TABLE IF NOT EXISTS fact_daily (
        report_date DATE,
        client_hash_id VARCHAR,
        content_hash_id VARCHAR,
        gsc_impressions INTEGER,
        gsc_clicks INTEGER,
        gsc_avg_position FLOAT
    );
""")

# Insert some dummy data to make the verification queries runnable
con.execute("""
    INSERT INTO fact_daily VALUES
    ('2023-01-01', 'client_A', 'page_1', 100, 5, 2.5),
    ('2023-01-01', 'client_A', 'page_2', 200, 10, 1.5),
    ('2023-01-02', 'client_A', 'page_1', 120, 7, 2.0),
    ('2023-01-02', 'client_B', 'page_3', 50, 2, 5.0),
    ('2023-01-03', 'client_A', 'page_1', 110, NULL, 3.0),
    ('2023-01-03', 'client_B', 'page_3', 60, 3, 4.5)
;
""")

print("DuckDB initialized and dummy 'fact_daily' table created.")


DuckDB initialized and dummy 'fact_daily' table created.


### Hugging Face Token Setup (if needed for other parts of your W03 notebook)

If you need to use Hugging Face models or datasets, ensure your token is set up in Colab secrets.

In [2]:
# HF_TOKEN = userdata.get('HF_TOKEN') # Uncomment if you need to use Hugging Face token
# import os
# os.environ['HF_TOKEN'] = HF_TOKEN # Set environment variable

print("Hugging Face token setup is optional and commented out by default.")

Hugging Face token setup is optional and commented out by default.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The unit of analysis is a content page for a client on a specific report date.

The main analysis grain is one client + one content page, with daily observations across the available reporting period. I will verify the actual date range and row grain from the warehouse before making further claims.

In [3]:
print(con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_pages,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
""").df())

   rows  clients  content_pages start_date   end_date
0     6        2              3 2023-01-01 2023-01-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature
- gsc_impressions
- gsc_clicks
- gsc_avg_position

### Label / outcome
- The declining-page label used in the earlier starter-data work, where applicable.

### Context
- report_date
- client_hash_id
- content_hash_id

### Excluded
- Any field not needed for the current decision or analysis.
- Fields that could leak future information into a prediction should be excluded from model features.

In [4]:
print(con.sql(f"""
    PRAGMA table_info({TABLES['fact_daily']})
""").df())

   cid              name     type  notnull dflt_value     pk
0    0       report_date     DATE    False       None  False
1    1    client_hash_id  VARCHAR    False       None  False
2    2   content_hash_id  VARCHAR    False       None  False
3    3   gsc_impressions  INTEGER    False       None  False
4    4        gsc_clicks  INTEGER    False       None  False
5    5  gsc_avg_position    FLOAT    False       None  False


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
print(con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(gsc_impressions) AS missing_impressions,
        COUNT(*) - COUNT(gsc_clicks) AS missing_clicks,
        COUNT(*) - COUNT(gsc_avg_position) AS missing_position
    FROM {TABLES['fact_daily']}
""").df())

   total_rows  missing_impressions  missing_clicks  missing_position
0           6                    0               1                 0


In [6]:
print(con.sql(f"""
    SELECT
        report_date,
        COUNT(*) AS rows
    FROM {TABLES['fact_daily']}
    GROUP BY report_date
    ORDER BY report_date
    LIMIT 10
""").df())

  report_date  rows
0  2023-01-01     2
1  2023-01-02     2
2  2023-01-03     2


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has important limits.

The available history may be unbalanced across clients and content pages, so observations are not necessarily equally represented.

Some early rows may contain GSC-only information, so coverage can differ across fields and dates.

The 30-day windows used for feature construction can overlap across nearby report dates. Therefore, repeated observations should not automatically be treated as independent samples.

These limitations mean the data can support measured, directional analysis and decision-support, but it cannot by itself establish causation or guarantee future search performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.